### Project: CIS*6530 Assignment 1 Submission 5 <br>
### Author: Myunghee Jung<br>
### Date: 2026-03-24 ~ 2026-04-03<br>
### Description: CNN Sequence


In [2]:
import pickle, numpy as np, torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import scipy.sparse
import time

# 1. Load Augumented Dataset

In [3]:
t0 = time.time()
with open('ml_data/manual_augmented_data_sparse.pkl', 'rb') as f:
    X1_aug, X2_aug, y_aug_raw, _, _ = pickle.load(f)

print(f"Augumented Dataset: {len(y_aug_raw)} samples")

Augumented Dataset: 378 samples


In [4]:
with open('ml_data/preprocessing_data.pkl', 'rb') as f:
    X1, X2, y, vectorizer1, vectorizer2 = pickle.load(f)

# Opcode count for first 5 samples
print("Opcode counts per APT sample:")
for i in range(min(5, X1.shape[0])):
    count = X1[i].nnz
    apt_label = y[i]  
    opcodes = vectorizer1.get_feature_names_out()[X1[i].nonzero()[1]]
    print(f"APT {apt_label}: {count} opcodes → {opcodes[:3]}...")

print("\nTotal opcode vocabulary (first 20):")
print(vectorizer1.get_feature_names_out()[:20])

Opcode counts per APT sample:
APT admin338: 1 opcodes → ['jmp']...
APT APT16: 1 opcodes → ['jmp']...
APT APT17: 10 opcodes → ['jmp' 'xor' 'call']...
APT APT19: 1 opcodes → ['jmp']...
APT APT1: 126 opcodes → ['jmp' 'xor' 'call']...

Total opcode vocabulary (first 20):
['aaa' 'aad' 'aam' 'aas' 'adc' 'adcx' 'add' 'addpd' 'addps' 'addsd'
 'addss' 'adox' 'aesdec' 'aesdeclast' 'aesenc' 'aesenclast' 'aesimc'
 'aeskeygenassist' 'and' 'andn']


# 2. Preprocessing

### Data Preparation Steps 

| # | Step | Code | Purpose |
|-------|----------|----------|-------------|
| 1 | Sampling | `n_samples = min(1200, len(y_aug_raw))` | Memory optimization (max 1200 samples) |
| 2 | Feature Truncation | `X1_aug[idx, :3000]` | Speed up training (top 3000 features each) |
| 3 | Concatenation | `np.hstack([X1, X2])` | Combine → 6000 features (1-gram + 2-gram) |
| 4 | Scaling | `StandardScaler()` | CNN gradient stability (mean=0, std=1) |
| 5 | Label Encoding | `LabelEncoder()` | 'APT28' → 0 |
| 6 | Train/Test Split | `test_size=0.2, stratify=y` | 80% train, 20% test|


In [12]:
n_samples = min(1200, len(y_aug_raw))
idx = np.random.choice(len(y_aug_raw), n_samples, replace=False)

X1 = X1_aug[idx, :3000].toarray() if scipy.sparse.issparse(X1_aug) else X1_aug[idx, :3000]
X2 = X2_aug[idx, :3000].toarray() if scipy.sparse.issparse(X2_aug) else X2_aug[idx, :3000]
X = np.hstack([X1, X2])

scaler = StandardScaler()
X = scaler.fit_transform(X)

le = LabelEncoder()
y = le.fit_transform(np.array(y_aug_raw)[idx])
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Data ready: {X.shape}")
print("Classes:", le.classes_[:5], "...")
print("y[:10]:", y[:10])
print("y_tr[:5]:", y_tr[:5])
print("y_te[:5]:", y_te[:5])

Data ready: (378, 1611)
Classes: ['APT1' 'APT16' 'APT17' 'APT19' 'APT28'] ...
y[:10]: [26  6 26 15  1  6 15 28 14  5]
y_tr[:5]: [29 28 31 31 19]
y_te[:5]: [ 1 27 20 28  7]


In [14]:
class TemporalCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=8, padding=4)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=8, padding=4)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(64, 16)
        self.fc2 = nn.Linear(16, n_classes)
    
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).squeeze(2)
        x = torch.relu(self.fc1(x))
        return torch.log_softmax(self.fc2(x), 1)

In [15]:
# 4. Model and data setup
model = TemporalCNN(len(le.classes_))
# optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001) 
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)      # Adam
criterion = nn.NLLLoss()

X_tr_t = torch.FloatTensor(X_tr).unsqueeze(1)
X_te_t = torch.FloatTensor(X_te).unsqueeze(1)
y_tr_t = torch.LongTensor(y_tr)

train_ds = torch.utils.data.TensorDataset(X_tr_t, y_tr_t)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)

print(f"Model ready: {len(le.classes_)} classes")

Model ready: 33 classes


In [16]:
# 5. Training (30 epochs, with loss/accuracy logging)
print("\nStarting training for McLaughlin CNN...")
for epoch in range(30):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

        _, preds_batch = torch.max(outputs, 1)
        correct += (preds_batch == batch_y).sum().item()
        total += batch_y.size(0)

    acc_tr = correct / total
    print(f"Epoch {epoch+1}/30 - Loss: {epoch_loss/len(train_loader):.4f}, Train Acc: {acc_tr:.2%}")


Starting training for McLaughlin CNN...
Epoch 1/30 - Loss: 3.5037, Train Acc: 3.31%
Epoch 2/30 - Loss: 3.4862, Train Acc: 3.31%
Epoch 3/30 - Loss: 3.4645, Train Acc: 4.30%
Epoch 4/30 - Loss: 3.4243, Train Acc: 3.64%
Epoch 5/30 - Loss: 3.3799, Train Acc: 2.98%
Epoch 6/30 - Loss: 3.3250, Train Acc: 2.98%
Epoch 7/30 - Loss: 3.2476, Train Acc: 6.62%
Epoch 8/30 - Loss: 3.1596, Train Acc: 7.95%
Epoch 9/30 - Loss: 3.0852, Train Acc: 5.96%
Epoch 10/30 - Loss: 3.0105, Train Acc: 9.60%
Epoch 11/30 - Loss: 2.9694, Train Acc: 7.28%
Epoch 12/30 - Loss: 2.9174, Train Acc: 7.62%
Epoch 13/30 - Loss: 2.8559, Train Acc: 7.62%
Epoch 14/30 - Loss: 2.8017, Train Acc: 9.93%
Epoch 15/30 - Loss: 2.7567, Train Acc: 10.26%
Epoch 16/30 - Loss: 2.7189, Train Acc: 11.59%
Epoch 17/30 - Loss: 2.6710, Train Acc: 9.93%
Epoch 18/30 - Loss: 2.6272, Train Acc: 15.89%
Epoch 19/30 - Loss: 2.5736, Train Acc: 16.56%
Epoch 20/30 - Loss: 2.5370, Train Acc: 16.23%
Epoch 21/30 - Loss: 2.5144, Train Acc: 21.19%
Epoch 22/30 - Los

In [17]:
# 6. Evaluation
model.eval()
with torch.no_grad():
    outputs = model(X_te_t)
    _, preds = torch.max(outputs, 1)
    accuracy = accuracy_score(y_te, preds.numpy())

print("\n" + "="*70)
print("="*70)
print("y_te[:10]:", y_te[:10])
print("preds[:10]:", preds.numpy()[:10])
print("y_te unique:", np.unique(y_te))
print("preds unique:", np.unique(preds.numpy()))

print(f"{'Method':<20} {'Accuracy'}")
print(f"{'1D CNN':<20} {accuracy:.1%}")
print(f"{'KNN 2-gram':<20} {'85.7%'}")
print(f"{'SVM 1-gram':<20} {'83.2%'}")
print(f"{'Improvement':<20} {+(accuracy*100-85.7):.1f}p")
print("-"*70)
print(f"Total time: {time.time()-t0:.1f} s")
print("="*70)


print("\nClass-wise performance:")
print(classification_report(y_te, preds.numpy(), target_names=le.classes_, zero_division=0))


y_te[:10]: [ 1 27 20 28  7 18 22 17 32 22]
preds[:10]: [18 27 29 28 18 18 18 17 18 18]
y_te unique: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32]
preds unique: [ 2  4  6  8 10 11 12 17 18 21 25 27 28 29 30]
Method               Accuracy
1D CNN               30.3%
KNN 2-gram           85.7%
SVM 1-gram           83.2%
Improvement          -55.4p
----------------------------------------------------------------------
Total time: 333.6 s

Class-wise performance:
                 precision    recall  f1-score   support

           APT1       0.00      0.00      0.00         3
          APT16       0.00      0.00      0.00         2
          APT17       1.00      1.00      1.00         2
          APT19       0.00      0.00      0.00         2
          APT28       0.00      0.00      0.00         3
          APT29       0.00      0.00      0.00         3
           APT3       0.50      1.00      0.67         2
          APT33       0.

In [18]:
# 7. Sanity check with Logistic Regression and Random Forest
print("\nLogistic Regression / Random Forest sanity check:")
lr = LogisticRegression(max_iter=1000)
rf = RandomForestClassifier(n_estimators=100)

lr.fit(X_tr, y_tr)
rf.fit(X_tr, y_tr)

print("LR test acc:", accuracy_score(y_te, lr.predict(X_te)))
print("RF test acc:", accuracy_score(y_te, rf.predict(X_te)))


Logistic Regression / Random Forest sanity check:
LR test acc: 0.8421052631578947
RF test acc: 0.8421052631578947


In [20]:
# 8. Save model
torch.save(model.state_dict(), 'cnn_sequence.pth')
print("\n Model saved as cnn_sequence.pth")


 Model saved as cnn_sequence.pth
